In [47]:
# !pip install -U sentence_transformers outlines langchain-classic langchain-community langchain-chroma langchain-core langchain-text-splitters requests rank_bm25 langchain_huggingface

In [48]:
import pandas as pd
import os
import ast
from datasets import Dataset
import torch
# device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
from datasets import load_dataset
from sentence_transformers.util import mine_hard_negatives
from sentence_transformers import CrossEncoder
from sentence_transformers import SentenceTransformer
from sentence_transformers.cross_encoder import (
    CrossEncoder,
    CrossEncoderModelCardData,
    CrossEncoderTrainer,
    CrossEncoderTrainingArguments,
)
from sentence_transformers.cross_encoder.evaluation import (
    CrossEncoderNanoBEIREvaluator,
    CrossEncoderRerankingEvaluator,
)
from sentence_transformers.cross_encoder.losses.BinaryCrossEntropyLoss import BinaryCrossEntropyLoss
from sentence_transformers.evaluation.SequentialEvaluator import SequentialEvaluator

In [49]:
def context_query_answers():
    base_data_path = '/content/drive/MyDrive/Colab Notebooks/RAG/Data'
    train_dataframe = pd.read_excel(os.path.join(base_data_path,'train.xlsx'))
    val_dataframe = pd.read_excel(os.path.join(base_data_path,'val.xlsx'))
    test_dataframe = pd.read_excel(os.path.join(base_data_path,'test.xlsx'))

    dataframe_list = [train_dataframe,val_dataframe,test_dataframe]
    query_list = []
    context_list = []
    answer_list = []

    for dataf in dataframe_list:
        for ind in range(0,len(dataf)):
            context_list.extend([dataf['context'][ind]])
            query_list.extend([dataf['question'][ind]])
            answer_list.extend([dataf['answers'][ind]])


    answer_list = [ast.literal_eval(i)[0] for i in answer_list]
    return context_list,query_list,answer_list


def sent_find(context,ans):

    ctx_list = [i for i in context.split("\n") if i]
    for ind , sen in enumerate(ctx_list):
        if ans in sen:
            if ind+1 < len(ctx_list):
                return ctx_list[ind]+ ' '+ ctx_list[ind+1]
            else:
                return ctx_list[ind]


    return None


def queries_anchor_creation(context_list,query_list,answer_list):
    anchor = []
    positive = []
    for ind , ans in enumerate(answer_list):
        sentence = sent_find(context_list[ind],ans)
        if sentence:
            anchor.append(query_list[ind])
            positive.append(sentence)
        else:
            print(ind)
            print("Not Fund")
    return anchor,positive

def custom_dataset_creation(threshold,val_thereshold):
    context_list,query_list,answer_list = context_query_answers()
    anchor,positive = queries_anchor_creation(context_list,query_list,answer_list)

    full_data_list = []
    train_data_list = []
    for ind , value in enumerate(anchor[:threshold]):
        mydict = {}
        mydict['question'] = value
        mydict['answer'] = positive[ind]
        train_data_list.append(mydict)
    train_dataset = Dataset.from_list(train_data_list)

    eval_data_list = []
    for ind , value in enumerate(anchor[threshold:threshold+val_thereshold]):
        mydict = {}
        mydict['question'] = value
        mydict['answer'] = positive[threshold+ind]
        eval_data_list.append(mydict)
    eval_dataset = Dataset.from_list(eval_data_list)

    full_data_list.extend(train_dataset)
    full_data_list.extend(eval_dataset)
    full_dataset = Dataset.from_list(full_data_list)

    return train_dataset,eval_dataset,full_dataset


In [50]:
train_datathreshold = 2000
val_data_thereshold = 500
train_dataset,eval_dataset,full_dataset = custom_dataset_creation(train_datathreshold,val_data_thereshold)

In [51]:
print(train_dataset)
print(eval_dataset)
print(full_dataset)

Dataset({
    features: ['question', 'answer'],
    num_rows: 2000
})
Dataset({
    features: ['question', 'answer'],
    num_rows: 500
})
Dataset({
    features: ['question', 'answer'],
    num_rows: 2500
})


In [52]:
print(train_dataset[0])

{'question': 'What was the amount of children murdered?', 'answer': 'The teen was one of 19 victims -- children and young women -- in one of the most gruesome serial killings in India in recent years. The Allahabad high court has acquitted Moninder Singh Pandher, his lawyer Sikandar B. Kochar told CNN.'}


In [53]:
print(eval_dataset[0])

{'question': 'What were police able to do for the first time?', 'answer': 'Police have been able to interview Brewer for the first time since the incident, the Broward County Sheriff\'s Office said. Hospital officials have said Brewer can communicate only in one- or two-word answers. "The more information we have, the better position we are in to make the right decision" as far as charges and how to proceed, said Maria Schneider, a prosecutor with the state attorney\'s office in Broward County.'}


### Hard Negative Data Mining using Embeeding Model

In [54]:
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
embedding_model = SentenceTransformer(model_name, device='cuda')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [55]:
### Hard Negative Mining for Train Data
num_hard_negatives = 5

hard_train_dataset = mine_hard_negatives(
    train_dataset,
    embedding_model,
    num_negatives=num_hard_negatives,  # How many negatives per question-answer pair
    absolute_margin=0,  # Similarity between query and negative samples should be x lower than query-positive similarity
    range_min=0,  # Skip the x most similar samples
    range_max=100,  # Consider only the x most similar samples
    sampling_strategy="top",  # Sample the top negatives from the range
    batch_size=16,  # Use a batch size of 4096 for the embedding model
    output_format="labeled-pair",  # The output format is (query, passage, label), as required by BinaryCrossEntropyLoss
    use_faiss=False,
)

Found 1997 unique queries out of 2000 total queries.
Found an average of 1.002 positives per query.


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/125 [00:00<?, ?it/s]

Negative candidates mined, preparing dataset...
Metric       Positive       Negative     Difference
Count           2,000          8,951               
Mean           0.4262         0.3617         0.0978
Median         0.4395         0.3549         0.0634
Std            0.1692         0.0956         0.0996
Min           -0.0637         0.0856         0.0000
25%            0.3041         0.2971         0.0139
50%            0.4395         0.3549         0.0634
75%            0.5536         0.4224         0.1601
Max            0.8415         0.7941         0.5536
Skipped 35,999 potential negatives (17.67%) due to the absolute_margin of 0.
Could not find enough negatives for 1049 samples (8.75%). Consider adjusting the range_max and absolute_margin parameters if you'd like to find more valid negatives.


In [56]:
### hard Negative Mining Using Eval Data for Evaluator

hard_eval_dataset = mine_hard_negatives(
        eval_dataset,
        embedding_model,
        num_negatives=30,  # How many documents to rerank
        batch_size=16,
        include_positives=True,
        output_format="n-tuple",
        use_faiss=False,
    )

Setting range_max to 31 based on the provided parameters.


Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Negative candidates mined, preparing dataset...
Metric       Positive       Negative     Difference
Count             500         15,000               
Mean           0.4323         0.2601         0.1722
Median         0.4415         0.2460         0.1719
Std            0.1739         0.0997         0.1811
Min           -0.0385         0.0468        -0.4706
25%            0.3079         0.1951         0.0314
50%            0.4416         0.2460         0.1719
75%            0.5643         0.3025         0.2982
Max            0.8505         0.8505         0.6782


#### Training Pipeline

In [57]:
#### Loading Cross Encoder Model for Reranker
model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2",device = 'cuda')

### Training Args
train_batch_size = 16
num_epochs = 10

print("Model max length:", model.max_length)
print("Model num labels:", model.num_labels)

### Defining Loss
loss = BinaryCrossEntropyLoss(model=model, pos_weight=torch.tensor(num_hard_negatives))

### Setting Evaluator
nano_beir_evaluator = CrossEncoderNanoBEIREvaluator(
        dataset_names=["msmarco", "nfcorpus", "nq"],
        batch_size=train_batch_size,
    )

reranking_evaluator = CrossEncoderRerankingEvaluator(
        samples=[
            {
                "query": sample["question"],
                "positive": [sample["answer"]],
                "documents": [sample[column_name] for column_name in hard_eval_dataset.column_names[2:]],
            }
            for sample in hard_eval_dataset
        ],
        batch_size=16,
        always_rerank_positives=False,
    )


### Evaluator Sequential combining reranking_evaluator, nano_beir_evaluator
evaluator = SequentialEvaluator([reranking_evaluator, nano_beir_evaluator])
evaluator(model)



Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

/tmp/ipykernel_6455/1044916019.py:8: DeprecationWarning: The `max_length` property was renamed and is now deprecated. Please use `max_seq_length` instead.
  print("Model max length:", model.max_length)


Model max length: 512
Model num labels: 1


{'map': 0.6956444983177742,
 'mrr@10': 0.6927373015873016,
 'ndcg@10': 0.7360874768665233,
 'base_map': 0.5679331792429464,
 'base_mrr@10': 0.5632388888888888,
 'base_ndcg@10': 0.6291665408834425,
 'NanoMSMARCO_R100_map': 0.603536741036741,
 'NanoMSMARCO_R100_mrr@10': 0.5962698412698413,
 'NanoMSMARCO_R100_ndcg@10': 0.6686056379547847,
 'NanoMSMARCO_R100_base_map': 0.4895766320756843,
 'NanoMSMARCO_R100_base_mrr@10': 0.4775,
 'NanoMSMARCO_R100_base_ndcg@10': 0.5404259879670522,
 'NanoNFCorpus_R100_map': 0.34607067087500665,
 'NanoNFCorpus_R100_mrr@10': 0.5885238095238094,
 'NanoNFCorpus_R100_ndcg@10': 0.39303074732957327,
 'NanoNFCorpus_R100_base_map': 0.2609949464238919,
 'NanoNFCorpus_R100_base_mrr@10': 0.49983333333333335,
 'NanoNFCorpus_R100_base_ndcg@10': 0.32504049401014556,
 'NanoNQ_R100_map': 0.7098410015737602,
 'NanoNQ_R100_mrr@10': 0.7355000000000002,
 'NanoNQ_R100_ndcg@10': 0.759868844367102,
 'NanoNQ_R100_base_map': 0.4196061957396544,
 'NanoNQ_R100_base_mrr@10': 0.4266904

In [58]:
### Evaluator reranking_evaluator
reranking_evaluator(model)

{'map': 0.6956444983177742,
 'mrr@10': 0.6927373015873016,
 'ndcg@10': 0.7360874768665233,
 'base_map': 0.5679331792429464,
 'base_mrr@10': 0.5632388888888888,
 'base_ndcg@10': 0.6291665408834425}

In [59]:
args = CrossEncoderTrainingArguments(
    # Required parameter:
    # Optional training parameters:
    num_train_epochs=num_epochs,
    per_device_train_batch_size=train_batch_size,
    per_device_eval_batch_size=train_batch_size,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=False,  # Set to False if you get an error that your GPU can't run on FP16
    bf16=True,  # Set to True if you have a GPU that supports BF16
    dataloader_num_workers=4,
    load_best_model_at_end=True,
    metric_for_best_model="ndcg@10",
    # Optional tracking/debugging parameters:
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    logging_steps=50,
    logging_strategy = 'steps',
    logging_first_step=True,
    dataloader_pin_memory=False,
    seed=12,
)

In [60]:
trainer = CrossEncoderTrainer(
    model=model,
    args=args,
    train_dataset=hard_train_dataset,
    loss=loss,
    evaluator=evaluator,
)


In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Step,Training Loss,Validation Loss,Map,Mrr@10,Ndcg@10,Base Map,Base Mrr@10,Base Ndcg@10,Nanomsmarco R100 Map,Nanomsmarco R100 Mrr@10,Nanomsmarco R100 Ndcg@10,Nanomsmarco R100 Base Map,Nanomsmarco R100 Base Mrr@10,Nanomsmarco R100 Base Ndcg@10,Nanonfcorpus R100 Map,Nanonfcorpus R100 Mrr@10,Nanonfcorpus R100 Ndcg@10,Nanonfcorpus R100 Base Map,Nanonfcorpus R100 Base Mrr@10,Nanonfcorpus R100 Base Ndcg@10,Nanonq R100 Map,Nanonq R100 Mrr@10,Nanonq R100 Ndcg@10,Nanonq R100 Base Map,Nanonq R100 Base Mrr@10,Nanonq R100 Base Ndcg@10,Nanobeir R100 Mean Map,Nanobeir R100 Mean Mrr@10,Nanobeir R100 Mean Ndcg@10,Nanobeir R100 Mean Base Map,Nanobeir R100 Mean Base Mrr@10,Nanobeir R100 Mean Base Ndcg@10,Sequential Score
100,2.146250,No log,0.695088,0.692242,0.736203,0.567933,0.563239,0.629167,0.588052,0.592079,0.669689,0.489577,0.477500,0.540426,0.346869,0.588524,0.393148,0.260995,0.499833,0.325040,0.709454,0.734833,0.759433,0.419606,0.426690,0.500647,0.548125,0.638479,0.607423,0.390059,0.468008,0.455371,0.607423
200,1.513191,No log,0.696103,0.693542,0.737726,0.567933,0.563239,0.629167,0.617896,0.612492,0.681614,0.489577,0.477500,0.540426,0.348530,0.597333,0.394600,0.260995,0.499833,0.325040,0.698695,0.716500,0.749771,0.419606,0.426690,0.500647,0.555040,0.642108,0.608662,0.390059,0.468008,0.455371,0.608662
300,1.338247,No log,0.705126,0.702779,0.746020,0.567933,0.563239,0.629167,0.621249,0.612770,0.680039,0.489577,0.477500,0.540426,0.349489,0.607524,0.397148,0.260995,0.499833,0.325040,0.704778,0.721333,0.754474,0.419606,0.426690,0.500647,0.558505,0.647209,0.610553,0.390059,0.468008,0.455371,0.610553
400,1.139366,No log,0.705547,0.704837,0.748302,0.567933,0.563239,0.629167,0.619927,0.613802,0.681197,0.489577,0.477500,0.540426,0.338710,0.608000,0.389886,0.260995,0.499833,0.325040,0.703807,0.718000,0.753229,0.419606,0.426690,0.500647,0.554148,0.646601,0.608104,0.390059,0.468008,0.455371,0.608104
500,0.828771,No log,0.709159,0.707663,0.751603,0.567933,0.563239,0.629167,0.613052,0.605881,0.674913,0.489577,0.477500,0.540426,0.335765,0.580667,0.384531,0.260995,0.499833,0.325040,0.672775,0.685333,0.730806,0.419606,0.426690,0.500647,0.540531,0.623960,0.596750,0.390059,0.468008,0.455371,0.596750
600,0.869838,No log,0.710033,0.708610,0.752205,0.567933,0.563239,0.629167,0.597048,0.590071,0.659242,0.489577,0.477500,0.540426,0.338796,0.609000,0.385187,0.260995,0.499833,0.325040,0.658809,0.671056,0.723621,0.419606,0.426690,0.500647,0.531551,0.623376,0.589350,0.390059,0.468008,0.455371,0.589350
700,0.772824,No log,0.714084,0.712271,0.754088,0.567933,0.563239,0.629167,0.590618,0.583389,0.654378,0.489577,0.477500,0.540426,0.335395,0.595857,0.382237,0.260995,0.499833,0.325040,0.654678,0.657556,0.718597,0.419606,0.426690,0.500647,0.526897,0.612267,0.585071,0.390059,0.468008,0.455371,0.585071
800,0.852260,No log,0.703086,0.701548,0.746176,0.567933,0.563239,0.629167,0.585802,0.577611,0.644349,0.489577,0.477500,0.540426,0.332172,0.587889,0.379444,0.260995,0.499833,0.325040,0.616711,0.621714,0.688326,0.419606,0.426690,0.500647,0.511562,0.595738,0.570706,0.390059,0.468008,0.455371,0.570706
900,0.651872,No log,0.714900,0.713740,0.756888,0.567933,0.563239,0.629167,0.574607,0.564746,0.631972,0.489577,0.477500,0.540426,0.332352,0.585048,0.376605,0.260995,0.499833,0.325040,0.582649,0.583190,0.652927,0.419606,0.426690,0.500647,0.496536,0.577661,0.553835,0.390059,0.468008,0.455371,0.553835
1000,0.730838,No log,0.707799,0.706942,0.749704,0.567933,0.563239,0.629167,0.575034,0.566857,0.636546,0.489577,0.477500,0.540426,0.334728,0.589270,0.380210,0.260995,0.499833,0.325040,0.584770,0.587190,0.663971,0.419606,0.426690,0.500647,0.498177,0.581106,0.560242,0.390059,0.468008,0.455371,0.560242


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
reranking_evaluator(model)